# sw1-1 - solid model with embedded bars

Specimen **SW1-1** (Lu, Xie, Guan, Huang & Lu 2015; test by Zhang, Tongji 2007): slender wall
1000 x 2000 x 125 with confined boundary elements, 246 kN axial. Built like `RW2_Wallace/rw2_solid_ladruno_embedment.ipynb`:
**hexahedra** (`LadrunoBrick`, formulation `brick`) and **discrete bars** embedded in the concrete (`LadrunoEmbeddedRebar`, perfect
bond, `CorotTruss` bars), specimen data (same values as `sw1_1_lu_specimen.ipynb`). Units: **N, mm, MPa**.

| item | here | source |
|---|---|---|
| wall | 1000 x 2000 x 125, boundary elements 200 mm | Lu2015 Fig. 7a |
| base block | 1600 x 550 x 125, one volume, elastic, E x (390 / 125) = in-plane stiffness of the 390 mm wide block | Fig. 7a; E baseline |
| concrete | `ASDConcrete3D` 9P: web 20.7 MPa at 0.002; boundary element core 25 MPa at 0.005 | THU Table 2; core baseline (Mander, no values) |
| boundary bars | 6 phi10 per end (3 columns x 2 faces) | Fig. 7a section 2-2 |
| web bars | 5 phi6 vertical per face, phi6 @ 125 horizontal | Fig. 7a |
| hoops | phi6 @ 80 around each group of 6 phi10, full height | Fig. 7a, THU Table 1 |
| steel | `Hysteretic`: phi6 392 / 2 % / 479 at 10 %; phi10 379 / 2 % / 554 at 10 %; b 0.01; pinching 0.5 / 0.8 | Fig. 7a; b THU deck; eu and pinching baseline |
| protocol | 24 reversal points of the test file, 0.25 mm increments | `sw1-1_Test.txt` |

Windows: runs with the Ladruno `OpenSeesMP` in **one process**. The installed apeGmsh cannot
export embedded-bar ties under partitioned MPI. Model inputs and loading history are unchanged.
Use the `C:\venv\opensees_venv\Scripts\python.exe` kernel. Evidence for every value: `context/`.

In [ ]:
import math
import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

from apeGmsh import apeGmsh, Results
from apeGmsh.opensees import apeSees
from apeGmsh.rebar import Cage

notebook_dir = Path(r'D:\OneDrive\Escritorio\AHORA SI\Notebooks\SW1-1_Lu')
opensees_mp = Path(r'C:\OpenSees-Solvers\opensees-bin\OpenSeesMP.exe')
launcher = Path(r'C:\OpenSees-Solvers\openseesmp.bat')
nparts = 1  # installed apeGmsh cannot export embedded bars under partitioned MPI
run_dir = notebook_dir / 'sw1_1_lu_solid_ladruno_embedment'
exp_file = notebook_dir / 'sw1-1_Test.txt'  # mm, kN (digitized curve)

for required in (opensees_mp, launcher, exp_file):
    if not required.is_file():
        raise FileNotFoundError(required)
run_dir.mkdir(parents=True, exist_ok=True)
print('Python:', sys.executable)
print('experiment:', exp_file)
print('run folder:', run_dir)


## inputs (N, mm, MPa)

In [ ]:
# geometry, Lu 2015 Fig. 7a
lw = 1000.0          # wall length
hw = 2000.0          # wall height, base to lateral load
tw = 125.0           # wall thickness (y direction, centred at y = 0)
lbe = 200.0          # boundary element width
max_size = 200.0     # maximum finite element size (same as the specimen shell models)
brick = 'ssp'        # LadrunoBrick formulation: std (= stdBrick) | bbar | ssp (stabilized one point, no bending locking) | eas
bar_refine = 4       # bar mesh = max_size / bar_refine (bars 4 times finer than the concrete)
base_overhang = 300.0
base_depth = 550.0
record_all = True    # True: every concrete node (viewer, isometric views); False: loaded nodes only (calibration runs)

# concrete, THUShell Table 2 (fc 20.7, ft 2.07, fcu 4.14, eps_c0 0.002), same as sw1_1_lu_specimen
ec = 21400.0                     # not reported: baseline value (= 4700 sqrt(fc))
nu = 0.2
ft = 2.07
fcp, ecp, fcr = 20.7, 0.002, 4.14
gt = 0.073 * fcp ** 0.18         # n/mm, fib model code 2010 (not reported)
fc0, gc = 5.0, 30.0              # not reported: baseline values
pscale_t = pscale_c = 0.3
# confined boundary element core: Mander model in Lu 2015, values not reported -> baseline values
fc0_core, fcp_core, fcr_core, ecp_core, gc_core = 10.0, 25.0, 20.0, 0.005, 60.0

# elastic base block (1600 x 550 x 390, Fig. 7a); E not reported: baseline value, scaled to t = tw
base_e = 23000.0 * 390.0 / tw

# bars, Fig. 7a; positions not dimensioned -> 15 mm cover to the hoop axis
db6, db10 = 6.0, 10.0
fy6, es6 = 392.0, 200600.0
fy10, es10 = 379.0, 202700.0
b_h = 0.01                       # hardening, THUShell example deck Steel02
fu6, fu10 = 479.0, 554.0           # Fig. 7a
# hardening shape not reported: calibrated stko 03_MITC law (390 / 540 at 2 % / 554 at 12 %), i.e. the stress
# at 2 % is 91.5 % of the way from fy to fu, applied to the measured fy and fu of each bar
k2, eu = (540.0 - 390.0) / (554.0 - 390.0), 0.12
s2_6 = fy6 + k2 * (fu6 - fy6)       # stress at 2 %
s2_10 = fy10 + k2 * (fu10 - fy10)
pinch_x, pinch_y = 0.5, 0.8      # baseline (stko 03_MITC)
edge = 23.0                      # phi10 axis to the face and to the wall end (15 cover + 3 hoop + 5)
be_cols = 3                      # phi10 columns per boundary element (3 x 2 faces = 6)
web_n, web_step = 5, 125.0       # 5 phi6 vertical per face @ 125
h_step = 125.0                   # phi6 horizontal @ 125
hoop_step = 80.0                 # phi6 hoops @ 80
anchor_depth = base_depth - 50.0 # vertical bars go this deep into the base
k_perfect = 1.0e8                # perfect-bond axial penalty of LadrunoEmbeddedRebar

# loads
p_axial = 246000.0               # n, Fig. 7a (0.095 f'c Ag)
gamma_c = 2.4e-9 * 9810.0        # n/mm3, self weight of the wall

print(f'axial {p_axial / 1e3:.1f} kn = {p_axial / (fcp * lw * tw):.3f} f\'c Ag | base e {base_e:.0f} mpa')
print(f'bar areas: phi6 {math.pi * db6 ** 2 / 4:.2f} | phi10 {math.pi * db10 ** 2 / 4:.2f} mm2')

## geometry

Same x-z layout as `sw1_1_lu`, built on the face y = -tw/2 and then extruded through the thickness: one
base rectangle (one physical group, one volume) and the wall rectangles (two boundary elements + web).
`fragment` cuts the base top edge under the wall and, with four points, its bottom edge at the same x, so
the base meshes as one structured patch. `extrude` with layers and `recombine` gives hexahedra.

| physical group | volumes | use |
|---|---|---|
| `Wall_BE` | boundary elements (full height, hoops @ 80 all the way up) | confined concrete |
| `Wall_Web` | web | unconfined concrete |
| `Base_Block` | base block | elastic |
| `Concrete` | all volumes | host of the embedded bars |
| `Fixed` | base bottom, both base sides, exposed base top | supports |
| `Top` | wall top face | axial load, imposed displacement |

In [ ]:
g = apeGmsh(model_name='sw1_1_lu_solid_ladruno_embedment', verbose=False)
g.begin()

x0, x1 = -base_overhang, lw + base_overhang
wall_x = [0.0, lbe, lw - lbe, lw]
y0 = -tw / 2                                 # extrusion face; the wall spans y -tw/2 .. tw/2

g.model.geometry.add_rectangle(x0, y0, -base_depth, x1 - x0, base_depth, plane='xz', label='base_block')
g.model.geometry.add_rectangle(0.0, y0, 0.0, lbe, hw, plane='xz', label='be_left')
g.model.geometry.add_rectangle(lbe, y0, 0.0, lw - 2 * lbe, hw, plane='xz', label='web')
g.model.geometry.add_rectangle(lw - lbe, y0, 0.0, lbe, hw, plane='xz', label='be_right')
bottom_pts = [g.model.geometry.add_point(x, y0, -base_depth) for x in wall_x]
g.model.boolean.fragment(['base_block'], ['be_left', 'web', 'be_right'] + [(0, p) for p in bottom_pts], dim=2)

ny = math.ceil(tw / max_size - 1e-9) + 1       # nodes through the thickness
g.model.transforms.extrude(g.model.select(dim=2).tags(), 0.0, tw, 0.0, num_elements=[ny - 1], recombine=True)

tol = 1e-3
y1 = tw / 2 + tol

def vols_in(lo, hi):
    return g.model.select(dim=3).in_box(lo, hi).tags()

def faces_in(lo, hi):
    return g.model.select(dim=2).in_box(lo, hi).tags()

g.physical.add_volume(vols_in((x0 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, tol)), name='Base_Block')
g.physical.add_volume(vols_in((-tol, y0 - tol, -tol), (lbe + tol, y1, hw + tol))
                      + vols_in((lw - lbe - tol, y0 - tol, -tol), (lw + tol, y1, hw + tol)), name='Wall_BE')
g.physical.add_volume(vols_in((lbe - tol, y0 - tol, -tol), (lw - lbe + tol, y1, hw + tol)), name='Wall_Web')
g.physical.add_volume(g.model.select(dim=3).tags(), name='Concrete')

fixed = (faces_in((x0 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, -base_depth + tol))   # base bottom
         + faces_in((x0 - tol, y0 - tol, -base_depth - tol), (x0 + tol, y1, tol))                # left side
         + faces_in((x1 - tol, y0 - tol, -base_depth - tol), (x1 + tol, y1, tol))                # right side
         + faces_in((x0 - tol, y0 - tol, -tol), (tol, y1, tol))                                  # exposed top, left
         + faces_in((lw - tol, y0 - tol, -tol), (x1 + tol, y1, tol)))                            # exposed top, right
g.physical.add_surface(fixed, name='Fixed')
g.physical.add_surface(faces_in((-tol, y0 - tol, hw - tol), (lw + tol, y1, hw + tol)), name='Top')

with g.loads.case('selfweight'):
    for grp in ('Wall_BE', 'Wall_Web'):
        g.loads.volume(grp, force_per_volume=(0.0, 0.0, -gamma_c))

with g.loads.case('axial'):
    g.loads.surface.traction('Top', vector=(0.0, 0.0, -p_axial / (lw * tw)))

## reinforcement (Fig. 7a)

Bar centres (x from the left wall end, y from the wall mid-plane). Positions are not dimensioned in
Lu2015; a 15 mm cover to the hoop axis is assumed.

- **boundary**: 6 phi10 per end, x = 23, 100, 177 (and mirrored), y = +-39.5.
- **web vertical**: 5 phi6 per face @ 125 centred on the wall, x = 250 ... 750, y = +-39.5.
- **web horizontal**: phi6 @ 125 up the height, x from 23 to 977, y = +-39.5 (same plane as the vertical
  bars, a truss idealisation; the U-shaped ends are not modelled).
- **hoops**: phi6 closed rectangles around each group of 6 phi10, @ 80 from z = 40 to the top; the inner
  tie line of Fig. 7a is not modelled.

Vertical bars run from `anchor_depth` inside the base to 23 mm below the top. All bars: `CorotTruss` on
their own line mesh, tied to the hexahedra by `LadrunoEmbeddedRebar` with perfect bond.

In [ ]:
yb = tw / 2 - edge                                    # 39.5 mm
be_x = [edge + j * (lbe - 2 * edge) / (be_cols - 1) for j in range(be_cols)]   # 23, 100, 177
web_x = [lw / 2 + (j - (web_n - 1) / 2) * web_step for j in range(web_n)]      # 250 ... 750
z_bot, z_top = -anchor_depth, hw - edge

bars = []

def add_bar(points, db, material, role):
    bars.append(g.rebar.bar(points, db=db, material=material, role=role, name=f'{role}_{len(bars)}'))

for y in (-yb, yb):
    for x in be_x + [lw - x for x in be_x]:
        add_bar([(x, y, z_bot), (x, y, z_top)], db10, 'steel_10', 'vertical')
    for x in web_x:
        add_bar([(x, y, z_bot), (x, y, z_top)], db6, 'steel_6', 'vertical')
    for k in range(1, int(z_top // h_step) + 1):
        z = k * h_step
        add_bar([(edge, y, z), (lw - edge, y, z)], db6, 'steel_6', 'horizontal')

off = (db10 + db6) / 2                                 # hoop centre line outside the bars
yh = yb + off
z = hoop_step / 2
while z < hw:
    for a, b in ((be_x[0] - off, be_x[-1] + off), (lw - be_x[-1] - off, lw - be_x[0] + off)):
        add_bar([(a, -yh, z), (b, -yh, z), (b, yh, z), (a, yh, z), (a, -yh, z)], db6, 'steel_6', 'tie')
    z += hoop_step

g.rebar.place(Cage(bars=tuple(bars)), into='Concrete', coupling='embedded', perfect=k_perfect,
              emit_elements=True)

n_role = {r: sum(b.role == r for b in bars) for r in ('vertical', 'horizontal', 'tie')}
print('bars:', n_role)

## mesh

Structured hexahedra. On the x-z face each piece gets `n = ceil(length / max_size) + 1` nodes per side
(same rule as `rw2_stko`), the base is one transfinite patch with its four corners given, and the
extrusion puts `ny - 1` layers through the thickness. The bars are meshed on their own at
`max_size / bar_refine` (4 times finer than the concrete). The Windows version keeps the mesh unpartitioned (`nparts = 1`) for embedded-bar export.

In [ ]:
def n_nodes(length):
    return math.ceil(length / max_size - 1e-9) + 1

nx_be, nx_web, nx_over = n_nodes(lbe), n_nodes(lw - 2 * lbe), n_nodes(base_overhang)
nz_wall, nz_base = n_nodes(hw), n_nodes(base_depth)
print(f'element size: be {lbe / (nx_be - 1):.1f} | web {(lw - 2 * lbe) / (nx_web - 1):.1f} | '
      f'thickness {tw / (ny - 1):.1f} | wall height {hw / (nz_wall - 1):.1f} | '
      f'overhang {base_overhang / (nx_over - 1):.1f} | base depth {base_depth / (nz_base - 1):.1f} mm')

g.mesh.recipe.structured('web', n=(nx_web, nz_wall), generate=False)
g.mesh.recipe.structured(['be_left', 'be_right'], n=(nx_be, nz_wall), generate=False)

def curves_in(lo, hi):
    return g.model.select(dim=1).in_box(lo, hi).tags()

def point_at(x, z):
    return g.model.select(dim=0).in_box((x - tol, y0 - tol, z - tol), (x + tol, y0 + tol, z + tol)).tags()[0]

spans = [(x0, 0.0, nx_over), (0.0, lbe, nx_be), (lbe, lw - lbe, nx_web), (lw - lbe, lw, nx_be), (lw, x1, nx_over)]
for z in (0.0, -base_depth):                       # base top and bottom edges, same divisions as the wall
    for a, b, n in spans:
        for c in curves_in((a - tol, y0 - tol, z - tol), (b + tol, y0 + tol, z + tol)):
            g.mesh.structured.set_transfinite_curve(c, n)
for x in (x0, x1):                                  # base sides
    for c in curves_in((x - tol, y0 - tol, -base_depth - tol), (x + tol, y0 + tol, tol)):
        g.mesh.structured.set_transfinite_curve(c, nz_base)
corners = [point_at(x0, -base_depth), point_at(x1, -base_depth), point_at(x1, 0.0), point_at(x0, 0.0)]
g.mesh.structured.set_transfinite_surface('base_block', corners=corners)
g.mesh.structured.set_recombine('base_block')
g.mesh.sizing.set_global_size(max_size / bar_refine)   # bar line mesh (the volumes are transfinite)

g.mesh.generation.generate(dim=3)
# Embedded-rebar MPI export is unsupported by the installed apeGmsh.
if nparts != 1:
    raise ValueError('This apeGmsh version requires nparts=1 for embedded bars')
fem = g.mesh.queries.get_fem_data(dim=None)

top_ids = [int(n) for n in fem.nodes.select(pg='Top').ids]
print(f'mesh : {fem.info.n_nodes} nodes | {fem.info.n_elems} elems | {nparts} partitions | types {fem.info.types}')
print(f'top nodes {len(top_ids)} | fixed nodes {len(fem.nodes.select(pg="Fixed").ids)} | '
      f'embedded ties {len(g.reinforce.reinforce_records)}')

## section cut

Set `z_cut` and run: plan view of the mesh at that height. Grey rectangles are the hexahedra cut by the
plane (dark: confined boundary elements, light: web); dots are bars crossing the plane (vertical bars, drawn
at their diameter); lines are bars lying in the plane within `dz` (horizontal bars blue, hoops red). Hoops
are at z = 40 + 80 k, horizontal bars at z = 125 k.

In [ ]:
concrete_colors = {'Wall_BE': '0.65', 'Wall_Web': '0.85', 'Base_Block': '0.95'}

In [ ]:
# section cut: concrete elements and bars at height z_cut
z_cut = 1000.0     # mm; 1000 has a hoop (40 + 12 x 80) and a horizontal bar (8 x 125)
dz = 5.0           # bars lying in the plane within +-dz are drawn as lines

xyz = dict(zip(fem.nodes.ids, fem.nodes.coords))

fig, ax = plt.subplots(figsize=(14, 3))
for grp, color in concrete_colors.items():
    for conn in fem.elements.select(pg=grp, element_type='hex8').connectivity:
        p = np.array([xyz[n] for n in conn])
        if p[:, 2].min() <= z_cut <= p[:, 2].max():
            ax.add_patch(plt.Rectangle((p[:, 0].min(), p[:, 1].min()), np.ptp(p[:, 0]), np.ptp(p[:, 1]),
                                       facecolor=color, edgecolor='0.4', lw=0.4))

bar_color = {'vertical': 'k', 'horizontal': 'tab:blue', 'tie': 'tab:red'}
for m in g.rebar.placements[0].members:
    for n1, n2 in fem.elements.select(pg=m.pg, element_type='line2').connectivity:
        a, b = np.array(xyz[n1]), np.array(xyz[n2])
        if abs(a[2] - z_cut) <= dz and abs(b[2] - z_cut) <= dz:              # bar in the cut plane
            ax.plot([a[0], b[0]], [a[1], b[1]], color=bar_color[m.role], lw=m.diameter / 3)
        elif min(a[2], b[2]) <= z_cut < max(a[2], b[2]):                     # bar crossing the cut
            t = (z_cut - a[2]) / (b[2] - a[2])
            p = a + t * (b - a)
            ax.add_patch(plt.Circle((p[0], p[1]), m.diameter / 2, color=bar_color[m.role]))

ax.set_aspect('equal')
ax.autoscale_view()
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(f'section at z = {z_cut:.0f} mm  (black: vertical bars, blue: horizontal bars, red: hoops)')
plt.show()

## mesh viewer

In [ ]:
g.mesh.viewer()

## opensees model

In [ ]:
ops = apeSees(fem)
ops.model(ndm=3, ndf=3)

def concrete(fcp, ecp, fc0, fcr, gc):
    # Current API: reproduce the same STKO Concrete (9P) law and IMPL-EX alpha=1.
    from apeGmsh.interop.stko.translate_props import asdconcrete_9p
    from apeGmsh.opensees.material.nd import ASDConcrete3D

    law = asdconcrete_9p(dict(
        E=ec, ft=ft, fc0=fc0, fcp=fcp, fcr=fcr, ecp=ecp, Gt=gt, Gc=gc,
        **{'PScale Tension': pscale_t, 'PScale Compression': pscale_c}))
    # The current material validates damage >= 0; suppress only round-off.
    for key in ('Td', 'Cd'):
        if min(law[key]) < -1e-12:
            raise ValueError(f'Invalid negative damage in {key}')
        law[key] = tuple(max(0.0, value) for value in law[key])
    return ops.register(ASDConcrete3D(E=ec, v=nu, implex=True, **law))

conc_web = concrete(fcp, ecp, fc0, fcr, gc)
conc_core = concrete(fcp_core, ecp_core, fc0_core, fcr_core, gc_core)
base_mat = ops.nDMaterial.ElasticIsotropic(E=base_e, nu=nu)

# bar materials: symmetric Hysteretic (yield, stress at 2 %, fu at eu), by the names used in g.rebar.bar
def hysteretic(fy, e_s, s2, fu, eu, name):
    ey = fy / e_s
    assert ey < 0.02 < eu and fy < s2 < fu, (fy, s2, fu, eu)
    return ops.uniaxialMaterial.Hysteretic(
        s1p=fy, e1p=ey, s2p=s2, e2p=0.02, s3p=fu, e3p=eu,
        s1n=-fy, e1n=-ey, s2n=-s2, e2n=-0.02, s3n=-fu, e3n=-eu,
        pinch_x=pinch_x, pinch_y=pinch_y, damage1=0.0, damage2=0.0, name=name)

hysteretic(fy6, es6, s2_6, fu6, eu, 'steel_6')       # phi6: web bars and hoops
hysteretic(fy10, es10, s2_10, fu10, eu, 'steel_10')  # phi10: boundary bars

ops.element.LadrunoBrick(pg='Wall_Web', material=conc_web, formulation=brick)
ops.element.LadrunoBrick(pg='Wall_BE', material=conc_core, formulation=brick)
ops.element.LadrunoBrick(pg='Base_Block', material=base_mat, formulation=brick)

ops.fix(pg='Fixed', dofs=(1, 1, 1))

## loading protocol (reversal points of the test, same as sw1_1_lu_specimen)

In [ ]:
peaks = [3.57, -3.39, 5.18, -6.07, 6.61, -7.86, 8.04, -9.46, 9.82, -10.9, 11.6, -12.5,
         13.2, -14.1, 14.6, -15.9, 16.6, -17.7, 18.6, -19.3, 20.2, -21.3, 20.2, -21.1]   # mm
disp_values = [0.0] + peaks
travel = np.cumsum(np.abs(np.diff(disp_values)))       # mm travelled up to each peak
time_values = [0.0] + [float(t) for t in travel / travel[-1]]   # pseudo-time proportional to travel
d_step = 0.25                                      # mm, displacement increment
n_cyclic = math.ceil(travel[-1] / d_step)
# hold the last peak a little past t = 1: the accumulated pseudo-time of the last increment can land a hair
# beyond 1.0, where the Path series returns 0 (a jump back to zero displacement)
disp_values.append(peaks[-1])
time_values.append(1.0 + 1.0 / n_cyclic)
print(f'{len(peaks)} peaks, {min(peaks):.2f} / {max(peaks):.2f} mm | travel {travel[-1]:.1f} mm | '
      f'{n_cyclic} increments of {travel[-1] / n_cyclic:.3f} mm')

## stages and deck

1. `gravity`: self weight + 246 kN axial as a traction on the top face, 10 increments, then held.
2. `cyclic`: `sp` ux on every top node following the path.

Windows single-process export: `RCM` numberer and `UmfPack` direct solver. One `.ladruno` recorder (displacement +
reaction of every node) in the cyclic stage. Base shear = sum of the x reactions of the top nodes.

`record_all = True` records every concrete node (results viewer, isometric views); `False` records only
the loaded nodes (small files, enough for the base shear curve; used in the calibration runs).

In [ ]:
rec = ops.recorder.Ladruno(file='sw1_1_lu_solid_ladruno_embedment.ladruno', nodal_responses=('displacement', 'reactionForce'),
                           nodes_pg='Concrete' if record_all else 'Top')

n_gravity = 10
dlam_gravity = 1.0 / n_gravity

with ops.stage(name='gravity') as s:
    with s.pattern(series=ops.timeSeries.Linear()) as p:
        p.from_model('selfweight')
        p.from_model('axial')
    s.analysis(test=ops.test.NormDispIncr(tol=1e-4, max_iter=50),
               algorithm=ops.algorithm.KrylovNewton(max_dim=10),
               integrator=ops.integrator.LoadControl(dlam=dlam_gravity),
               constraints=ops.constraints.Transformation(),
               numberer=ops.numberer.RCM(),
               system=ops.system.UmfPack(),
               analysis=ops.analysis.Static())
    s.run(n_increments=n_gravity, dt=dlam_gravity)

dlam_cyclic = 1.0 / n_cyclic

with ops.stage(name='cyclic') as s:
    with s.pattern(series=ops.timeSeries.Path(values=disp_values, time=time_values)) as p:
        p.sp(pg='Top', dof=1, value=1.0)
    s.recorder(rec)
    s.analysis(test=ops.test.NormDispIncr(tol=1e-4, max_iter=100),
               algorithm=ops.algorithm.KrylovNewton(max_dim=10),
               integrator=ops.integrator.LoadControl(dlam=dlam_cyclic),
               constraints=ops.constraints.Transformation(),
               numberer=ops.numberer.RCM(),
               system=ops.system.UmfPack(),
               analysis=ops.analysis.Static())
    # on a failed increment escalate ModifiedNewton(initial) -> KrylovNewton -> NewtonLineSearch
    s.run(n_increments=n_cyclic, dt=dlam_cyclic, strategy=ops.strategy.profile('penalty-stiff'))

ops.tcl(str(run_dir / 'sw1_1_lu_solid_ladruno_embedment.tcl'))
g.end()
print('deck:', run_dir / 'sw1_1_lu_solid_ladruno_embedment.tcl')

## run (Windows OpenSeesMP, one process)

The Windows launcher initializes the solver environment. The cell displays progress and a status
message every 30 seconds without new progress. On failure or interruption, only this run and its
child processes are stopped.


In [ ]:
deck_file = run_dir / 'sw1_1_lu_solid_ladruno_embedment.tcl'
if not deck_file.is_file():
    raise FileNotFoundError(f'Generate the deck first: {deck_file}')
if nparts != 1:
    raise ValueError('This apeGmsh version requires nparts=1 for embedded bars')

for f in run_dir.glob('sw1_1_lu_solid_ladruno_embedment*.ladruno'):   # clean HDF5 recorders on rerun
    f.unlink()

log_file = run_dir / 'sw1_1_lu_solid_ladruno_embedment_run.log'
failed = False
started = time.monotonic()
last_update = started
last_progress = ''
print(f'Starting OpenSees: {deck_file}', flush=True)
# With no second argument, this Windows launcher runs OpenSeesMP in one process.
# It also initializes Intel MPI, Tcl and the solver DLL search paths.
with log_file.open('w', encoding='utf-8') as log:
    proc = subprocess.Popen(
        [os.environ.get('COMSPEC', 'cmd.exe'), '/d', '/c', str(launcher), str(deck_file)],
        cwd=run_dir, stdout=log, stderr=subprocess.STDOUT,
        creationflags=subprocess.CREATE_NO_WINDOW)
    try:
        while proc.poll() is None:
            try:
                proc.wait(timeout=10)
            except subprocess.TimeoutExpired:
                pass
            text = log_file.read_text(encoding='utf-8', errors='replace')
            progress = [line for line in text.splitlines() if 'APEGMSH_PROGRESS' in line]
            elapsed = time.monotonic() - started
            if progress and progress[-1] != last_progress:
                last_progress = progress[-1]
                print(f'[{elapsed / 60:.1f} min] {last_progress}', flush=True)
                last_update = time.monotonic()
            elif time.monotonic() - last_update >= 30:
                print(f'[{elapsed / 60:.1f} min] OpenSees is running; '
                      f'last progress: {last_progress or "waiting for first increment"}', flush=True)
                last_update = time.monotonic()
            if any(token in text for token in ('FAILED', 'ERROR', 'FATAL', 'while executing')):
                failed = True
                break
    finally:
        if proc.poll() is None:   # this run only, including children; also handles kernel interrupts
            subprocess.run(['taskkill', '/PID', str(proc.pid), '/T', '/F'],
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                           creationflags=subprocess.CREATE_NO_WINDOW, check=False)
            proc.wait(timeout=30)

text = log_file.read_text(encoding='utf-8', errors='replace')
failed = failed or any(token in text for token in ('FAILED', 'ERROR', 'FATAL', 'while executing'))
print('return code:', proc.returncode, '| failed:', failed)
print([line for line in text.splitlines() if 'APEGMSH_PROGRESS' in line][-1:])
assert not failed and proc.returncode == 0, f'opensees run failed, see {log_file}'
assert f'APEGMSH_PROGRESS i={n_cyclic} n={n_cyclic} ' in text, f'Incomplete cyclic analysis, see {log_file}'


## results vs experiment

| source | file units | conversion |
|---|---|---|
| model (`.ladruno`, one file per rank, merged by `Results.from_ladruno`) | disp mm, reaction N | sum of the x reactions of the loaded nodes / 1000 |
| experiment (`sw1-1_Test.txt`) | disp mm, force kN | none |

In [ ]:
import h5py


def _has_nodes(path):
    with h5py.File(path, 'r') as f:
        return any(isinstance(g, h5py.Group) and 'MODEL/NODES' in g for g in f.values())


name = 'sw1_1_lu_solid_ladruno_embedment'
# a rank that owns none of the recorded nodes writes an empty file: read only the files with data
parts = [str(p) for p in sorted(run_dir.glob(f'{name}*.ladruno')) if _has_nodes(p)]
results = Results.from_ladruno(parts, merge_partitions=True)

disp = results.nodes.get(ids=top_ids, component='displacement_x').values[:, 0]                    # mm
shear = results.nodes.get(ids=top_ids, component='reaction_force_x').values.sum(axis=1) / 1000.0  # kn
n = min(len(disp), len(shear))
disp, shear = disp[:n], shear[:n]

exp = np.loadtxt(exp_file)
exp_disp, exp_shear = exp[:, 0], exp[:, 1]   # mm, kn

for label, d, v in (('solid', disp, shear), ('experiment', exp_disp, exp_shear)):
    print(f'{label:10s}: {len(d):5d} pts | disp {d.min():7.1f} / {d.max():6.1f} mm | shear {v.min():8.1f} / {v.max():7.1f} kn')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(exp_disp, exp_shear, 'k--', lw=1.0, label='experiment')
ax.plot(disp, shear, lw=1.0, label=f'solid, embedded bars (max_size {max_size:.0f})')
ax.set_xlabel('top displacement (mm)')
ax.set_ylabel('base shear (kN)')
ax.set_title('SW1-1 solid - base shear vs top displacement')
ax.grid(True)
ax.legend()
plt.show()

## results viewer

In [ ]:
results.viewer()

## results only (run from here)

Stand-alone: imports, reads this notebook's deck and `.ladruno` files from the last run and plots against the experiment. No need to run the cells above.

In [ ]:
# results only: run this cell by itself (fresh kernel is fine): it reads the deck and the .ladruno of the last run
import re
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from apeGmsh import Results
import h5py


def _has_nodes(path):
    with h5py.File(path, 'r') as f:
        return any(isinstance(g, h5py.Group) and 'MODEL/NODES' in g for g in f.values())


name = 'sw1_1_lu_solid_ladruno_embedment'
notebook_dir = Path(r'D:\OneDrive\Escritorio\AHORA SI\Notebooks\SW1-1_Lu')
run_dir = notebook_dir / name
exp_file = notebook_dir / 'sw1-1_Test.txt'   # mm, kN

deck = (run_dir / f'{name}.tcl').read_text(encoding='utf-8')
top_ids = sorted({int(n) for n in re.findall(r'^\s*sp (\d+) 1 ', deck, re.M)})   # nodes with the imposed displacement

# a rank that owns none of the recorded nodes writes an empty file: read only the files with data
parts = [str(p) for p in sorted(run_dir.glob(f'{name}*.ladruno')) if _has_nodes(p)]
results = Results.from_ladruno(parts, merge_partitions=True)

disp = results.nodes.get(ids=top_ids, component='displacement_x').values[:, 0]                    # mm
shear = results.nodes.get(ids=top_ids, component='reaction_force_x').values.sum(axis=1) / 1000.0  # kn
n = min(len(disp), len(shear))
disp, shear = disp[:n], shear[:n]

exp = np.loadtxt(exp_file)
exp_disp, exp_shear = exp[:, 0], exp[:, 1]   # mm, kn

for label, d, v in ((name, disp, shear), ('experiment', exp_disp, exp_shear)):
    print(f'{label:20s}: {len(d):5d} pts | disp {d.min():7.1f} / {d.max():6.1f} mm | shear {v.min():8.1f} / {v.max():7.1f} kn')

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(exp_disp, exp_shear, 'k--', lw=1.0, label='experiment')
ax.plot(disp, shear, lw=1.0, label=name)
ax.set_xlabel('top displacement (mm)')
ax.set_ylabel('base shear (kN)')
ax.set_title('SW1-1 - base shear vs top displacement')
ax.grid(True)
ax.legend()
plt.show()